In [0]:
payment_methods_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__payment_methods/"
)

payment_methods_df = (
    spark.read
    .option("multiLine", "false")
    .json(payment_methods_path + "*.jsonl.gz")
)

print("Total rows:", payment_methods_df.count())
print("Total columns:", len(payment_methods_df.columns))

payment_methods_df.printSchema()

In [0]:
# =====================================================
# Payment methods cardinality per listing
# =====================================================

from pyspark.sql import functions as F

payment_counts = (
    payment_methods_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("payment_method_count")
    )
)

display(
    payment_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("payment_method_count").alias("avg_methods"),
        F.expr("percentile_approx(payment_method_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(payment_method_count, 0.50)").alias("median"),
        F.expr("percentile_approx(payment_method_count, 0.75)").alias("p75"),
        F.min("payment_method_count").alias("min_methods"),
        F.max("payment_method_count").alias("max_methods")
    )
)

In [0]:
# =====================================================
# Validate payment-method → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

payment_parent_ids = (
    payment_methods_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    payment_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    payment_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Payment-method parent IDs:", payment_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# Payment method distribution
# =====================================================

display(
    payment_methods_df
    .groupBy("payment_method_type")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy(F.desc("record_count"))
)

In [0]:
# =====================================================
# Payment method quality
# =====================================================

payment_methods_df.agg(
    F.sum(
        F.when(F.col("payment_method_type").isNull(), 1)
         .otherwise(0)
    ).alias("null_payment_method"),

    F.sum(
        F.when(
            F.col("payment_method_type").isNotNull() &
            (F.trim(F.col("payment_method_type")) == ""),
            1
        ).otherwise(0)
    ).alias("blank_payment_method")
).show()

### Payment Methods Profile

The `payment_methods` dataset contains **199,051 records** across 4 columns:

- `_dlt_id`
- `_dlt_list_idx`
- `_dlt_parent_id`
- `payment_method_type`

The table represents the payment methods available for individual Item Details listings. `_dlt_parent_id` links each payment-method record back to its parent listing.

### Payment Methods per Listing

There are **67,877 listings** with payment-method records.

| Metric | Result |
|---|---:|
| Listings with payment methods | 67,877 |
| Average methods per listing | 2.9325 |
| P25 | 3 |
| Median | 3 |
| P75 | 3 |
| Minimum methods | 1 |
| Maximum methods | 6 |

Most listings have around **3 payment methods**, as both the median and the 25th/75th percentiles are 3. The observed range is **1 to 6 payment methods per listing**, confirming that payment methods form a multi-valued child dataset.

### Payment Methods → Item Details Relationship

| Validation | Result |
|---|---:|
| Payment-method parent IDs | 67,877 |
| Matched to current Item Details | 67,877 |
| Not found in current Item Details | 0 |

All **67,877 payment-method parent IDs** successfully matched an existing Item Details record.

There are therefore **0 orphaned payment-method records** in the current snapshot.

The relationship is:

```text
Item Details
    │
    │ 1
    │
    └──────────< Payment Methods
                   many (1–6 observed)
```

### Payment Method Distribution

| Payment Method | Record Count | Listing Count |
|---|---:|---:|
| `CREDIT_CARD` | 67,876 | 67,876 |
| `WALLET` | 67,876 | 67,876 |
| `OTHER` | 61,427 | 61,427 |
| `MONEY_ORDER` | 1,715 | 1,715 |
| `CASH` | 114 | 114 |
| `CHECK` | 38 | 38 |
| `WIRE_TRANSFER` | 5 | 5 |

`CREDIT_CARD` and `WALLET` are the most widely observed payment methods, each appearing for 67,876 listings.

`OTHER` is also common, appearing for 61,427 listings, while `MONEY_ORDER`, `CASH`, `CHECK`, and `WIRE_TRANSFER` occur much less frequently.

The `record_count` and `listing_count` values are identical for every payment-method type, indicating that each payment-method record represents one listing-method association in the current snapshot.


### Payment Method Data Quality

| Check | Result |
|---|---:|
| NULL `payment_method_type` | 0 |
| Blank / whitespace-only `payment_method_type` | 0 |

All payment-method records contain a populated and non-blank `payment_method_type`.

No additional cleaning is required for missing or whitespace-only `payment-method` values at this stage.

### Findings
The `payment_methods` dataset is a clean multi-valued child dataset of Item Details.

The parent relationship is complete, with 0 unmatched parent IDs, and each listing has between 1 and 6 observed payment methods.

The observed payment-method vocabulary is relatively small:

- `CREDIT_CARD`
- `WALLET`
- `OTHER`
- `MONEY_ORDER`
- `CASH`
- `CHECK`
- `WIRE_TRANSFER`

For the Silver layer, `_dlt_parent_id` can be retained as the relationship to Item Details and `payment_method_type` can be treated as a categorical field.

The dataset should remain as a separate child table, rather than being flattened into multiple payment-method columns in the main Item Details table.